# scExpand: machine-learning example

scExpand converts repeated within-cluster cell subsamples into sample-level
expression vectors. Inputs may come from CyTOF, scRNA-seq, or related annotated
single-cell measurements. This notebook runs the functions in `machine_learning.py`.
It is not tied to a particular disease, timepoint, feature count, or marker panel.

Source and citation: https://github.com/WeiJiang-ZJU/scExpand

The notebook ships without saved outputs or study data. The default example is
artificial and does not demonstrate clinical or sequencing performance.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
assert (ROOT / "machine_learning.py").exists(), "Open this notebook from the repository directory."
sys.path.insert(0, str(ROOT))
from machine_learning import make_demo_table, run_analysis

## Choose input data

The next cell creates an artificial demonstration table. For your own analysis,
remove the `make_demo_table` call and point `INPUT_CSV` to the CSV exported by
`scExpand.R`. `Outcome` must be 0/1 (or ALI/non-ALI); `Patient` identifies the
original donor or other independent biological unit. `Time` is optional when
only one timepoint is present. All nonmetadata columns are used as features.

In [ ]:
INPUT_CSV = ROOT / "demo_data.csv"
make_demo_table(INPUT_CSV, seed=24)  # Artificial data only; remove for real inputs.
TIMEPOINT = None  # Use a Time value if the CSV contains multiple timepoints.
FEATURE_FILE = None  # Optional subset: one exact feature name per line.

## Run the analysis

All pseudo-samples from one donor stay together in every split, including LR's
inner folds. Models are chosen by mean training-CV donor AUC. The selected model
is evaluated on held-out donors after averaging their pseudo-sample scores.
Default thresholds are fixed at 0.5 for probabilities and 0 for decision margins.
Feature selection and preprocessing must not use held-out outcomes.

In [ ]:
import uuid

OUTPUT_DIR = ROOT / "results" / ("run_" + uuid.uuid4().hex[:8])
metrics = run_analysis(
    INPUT_CSV, TIMEPOINT, OUTPUT_DIR,
    feature_file=FEATURE_FILE,
    models=None,  # None compares all 15 classifiers.
    seed=24, cv_folds=5, inner_folds=3,
)
metrics

## Inspect outputs

Candidate warnings and failures are retained in the comparison table. Results
and figures from the artificial example have no biological interpretation.
Pseudo-samples do not create new independent donors. Clear all outputs before
sharing a notebook run on private data.

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(OUTPUT_DIR / "cv_comparison.csv"))
for filename in ["roc.png", "precision_recall.png", "confusion_matrix.png"]:
    display(Image(filename=str(OUTPUT_DIR / filename)))